In [ ]:
"""
================================================================================
MONKEYPOX SEVERITY CLASSIFICATION - COMPLETE TRAINING WITH VISUALIZATIONS
================================================================================
Dataset: Google Drive - mpoxdataset/
- 1_Macules, 2_Papules, 3_Vesicles, 4_Pustules, 5_Scubs, 6_Normal
- Automatic 70/15/15 split
- Training and validation graphs
- Accuracy, Precision, Recall, F1 Score visualization
================================================================================
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import LabelEncoder
from scipy import stats
from tqdm import tqdm
import warnings
import glob
from google.colab import drive
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("=" * 80)
print("MONKEYPOX SEVERITY CLASSIFICATION - TRAINING PIPELINE")
print("=" * 80)
print(f"TensorFlow Version: {tf.__version__}")
print("=" * 80)

# ============================================================================
# PART 1: MOUNT GOOGLE DRIVE AND LOAD DATASET
# ============================================================================

print("\n[1] Mounting Google Drive...")
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/mpoxdataset/'

if not os.path.exists(DATASET_PATH):
    print(f"Error: Dataset not found at {DATASET_PATH}")
    print("Please ensure your dataset is in the correct location:")
    print("  /content/drive/MyDrive/mpoxdataset/")
    print("  ├── 1_Macules/")
    print("  ├── 2_Papules/")
    print("  ├── 3_Vesicles/")
    print("  ├── 4_Pustules/")
    print("  ├── 5_Scubs/")
    print("  └── 6_Normal/")
    sys.exit(1)

class_names = ['Macules', 'Papules', 'Vesicles', 'Pustules', 'Scubs', 'Normal']
class_mapping = {
    '1_Macules': 0,
    '2_Papules': 1,
    '3_Vesicles': 2,
    '4_Pustules': 3,
    '5_Scubs': 4,
    '6_Normal': 5
}

print("\n[2] Loading Dataset...")
print(f"Dataset Path: {DATASET_PATH}")

images = []
labels = []
image_paths = []

for folder_name, label in class_mapping.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)

    if not os.path.exists(folder_path):
        print(f"Warning: Folder not found: {folder_path}")
        continue

    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        image_files.extend(glob.glob(os.path.join(folder_path, ext)))

    print(f"  {folder_name}: {len(image_files)} images")

    for img_path in tqdm(image_files, desc=f"Loading {folder_name}"):
        try:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))
                images.append(img)
                labels.append(label)
                image_paths.append(img_path)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

images = np.array(images, dtype=np.float32) / 255.0
labels = np.array(labels)

print(f"\nDataset Loaded Successfully!")
print(f"  Total Images: {len(images)}")
print(f"  Image Shape: {images[0].shape}")
print(f"  Classes: {len(np.unique(labels))}")
print("\nClass Distribution:")
for i, class_name in enumerate(class_names):
    count = np.sum(labels == i)
    print(f"  {class_name}: {count} images ({count/len(labels)*100:.1f}%)")

# ============================================================================
# PART 3: DATA SPLITTING (70/15/15)
# ============================================================================

print("\n[3] Splitting Dataset (70% Train, 15% Validation, 15% Test)...")

# First split: 70% train, 30% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.30, stratify=labels, random_state=42
)

# Second split: 50% of temp for validation, 50% for test (15% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"\nSplit Summary:")
print(f"  Training samples:   {len(X_train)} ({len(X_train)/len(images)*100:.1f}%)")
print(f"  Validation samples: {len(X_val)} ({len(X_val)/len(images)*100:.1f}%)")
print(f"  Test samples:       {len(X_test)} ({len(X_test)/len(images)*100:.1f}%)")

print("\nClass Distribution in Splits:")
print("-" * 60)
print(f"{'Class':<12} {'Train':<8} {'Val':<8} {'Test':<8} {'Total':<8}")
print("-" * 60)
for i, class_name in enumerate(class_names):
    train_count = np.sum(y_train == i)
    val_count = np.sum(y_val == i)
    test_count = np.sum(y_test == i)
    total = train_count + val_count + test_count
    print(f"{class_name:<12} {train_count:<8} {val_count:<8} {test_count:<8} {total:<8}")

# ============================================================================
# PART 4: THRESHOLD SEGMENTATION
# ============================================================================

class ThresholdSegmentation:
    """Threshold segmentation for isolating skin lesions."""

    def __init__(self, method='adaptive'):
        self.method = method

    def segment_image(self, image):
        """Apply threshold segmentation to isolate lesions."""
        if len(image.shape) == 3:
            gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        else:
            gray = (image * 255).astype(np.uint8)

        blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        if self.method == 'otsu':
            _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif self.method == 'adaptive':
            mask = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                        cv2.THRESH_BINARY, 11, 2)
        else:
            _, mask = cv2.threshold(blurred, 128, 255, cv2.THRESH_BINARY)

        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        if len(image.shape) == 3:
            segmented = cv2.bitwise_and((image * 255).astype(np.uint8),
                                       (image * 255).astype(np.uint8), mask=mask)
            segmented = segmented.astype(np.float32) / 255.0
        else:
            segmented = cv2.bitwise_and((image * 255).astype(np.uint8),
                                       (image * 255).astype(np.uint8), mask=mask)
            segmented = segmented.astype(np.float32) / 255.0

        return segmented

print("\n[4] Applying Threshold Segmentation...")
segmenter = ThresholdSegmentation(method='adaptive')

print("  Segmenting training set...")
X_train_seg = np.array([segmenter.segment_image(img) for img in tqdm(X_train, desc="  Training")])

print("  Segmenting validation set...")
X_val_seg = np.array([segmenter.segment_image(img) for img in tqdm(X_val, desc="  Validation")])

print("  Segmenting test set...")
X_test_seg = np.array([segmenter.segment_image(img) for img in tqdm(X_test, desc="  Test")])

# ============================================================================
# PART 5: CNN MODEL ARCHITECTURE
# ============================================================================

class MonkeypoxSeverityCNN:
    """Deep CNN for monkeypox severity classification."""

    def __init__(self, input_shape=(224, 224, 3), num_classes=6):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = None

    def build_model(self):
        """Build the CNN model."""
        inputs = layers.Input(shape=self.input_shape)

        # First block
        x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.25)(x)

        # Second block
        x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.25)(x)

        # Third block
        x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.3)(x)

        # Fourth block
        x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.3)(x)

        # Fifth block
        x = layers.Conv2D(512, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(512, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.5)(x)

        # Dense layers
        x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)

        # Output
        outputs = layers.Dense(self.num_classes, activation='softmax')(x)

        self.model = models.Model(inputs=inputs, outputs=outputs)
        return self.model

    def compile_model(self, learning_rate=0.001):
        """Compile the model with metrics."""
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return self.model

# ============================================================================
# PART 6: TRAINING FUNCTION WITH CALLBACKS
# ============================================================================

print("\n[5] Building and Training Model...")

# Build model
cnn = MonkeypoxSeverityCNN(input_shape=(224, 224, 3), num_classes=len(class_names))
model = cnn.build_model()
model = cnn.compile_model(learning_rate=0.001)

# Model summary
model.summary()

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Train model
print("\n[6] Training Model...")
history = model.fit(
    X_train_seg, y_train,
    validation_data=(X_val_seg, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# ============================================================================
# PART 7: EVALUATION
# ============================================================================

print("\n[7] Evaluating Model...")

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_seg, y_test, verbose=0)

# Predictions
y_pred_proba = model.predict(X_test_seg)
y_pred = np.argmax(y_pred_proba, axis=1)

# Classification report
report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)

# Calculate metrics
accuracy = test_accuracy * 100
precision = np.mean([report[c]['precision'] for c in class_names]) * 100
recall = np.mean([report[c]['recall'] for c in class_names]) * 100
f1 = np.mean([report[c]['f1-score'] for c in class_names]) * 100

print(f"\n{'='*60}")
print("FINAL PERFORMANCE METRICS")
print(f"{'='*60}")
print(f"Test Accuracy:    {accuracy:.2f}%")
print(f"Average Precision: {precision:.2f}%")
print(f"Average Recall:    {recall:.2f}%")
print(f"Average F1-Score:  {f1:.2f}%")
print(f"{'='*60}")

# ============================================================================
# PART 8: TRAINING AND VALIDATION GRAPHS
# ============================================================================

print("\n[8] Generating Training and Validation Graphs...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Training and Validation Accuracy
ax1 = axes[0, 0]
ax1.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2, color='blue')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='red')
ax1.axhline(y=test_accuracy, color='green', linestyle='--', label=f'Test Accuracy: {test_accuracy*100:.2f}%', linewidth=2)
ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, len(history.history['accuracy']))
ax1.set_ylim(0, 1.0)

# Add best accuracy annotation
best_val_acc = max(history.history['val_accuracy'])
best_epoch = history.history['val_accuracy'].index(best_val_acc)
ax1.annotate(f'Best: {best_val_acc*100:.2f}%\nEpoch {best_epoch+1}',
            xy=(best_epoch, best_val_acc),
            xytext=(best_epoch + 10, best_val_acc - 0.1),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=10, color='red')

# Plot 2: Training and Validation Loss
ax2 = axes[0, 1]
ax2.plot(history.history['loss'], label='Training Loss', linewidth=2, color='blue')
ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='red')
ax2.axhline(y=test_loss, color='green', linestyle='--', label=f'Test Loss: {test_loss:.4f}', linewidth=2)
ax2.set_xlabel('Epochs', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, len(history.history['loss']))
ax2.set_ylim(0, max(max(history.history['loss']), max(history.history['val_loss'])) * 1.1)

# Add best loss annotation
best_val_loss = min(history.history['val_loss'])
best_loss_epoch = history.history['val_loss'].index(best_val_loss)
ax2.annotate(f'Best: {best_val_loss:.4f}\nEpoch {best_loss_epoch+1}',
            xy=(best_loss_epoch, best_val_loss),
            xytext=(best_loss_epoch - 15, best_val_loss + 0.3),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=10, color='red')

# Plot 3: Combined Accuracy and Loss
ax3 = axes[1, 0]
ax3_twin = ax3.twinx()
ax3.plot(history.history['accuracy'], label='Train Acc', linewidth=2, color='blue', alpha=0.7)
ax3.plot(history.history['val_accuracy'], label='Val Acc', linewidth=2, color='red', alpha=0.7)
ax3_twin.plot(history.history['loss'], label='Train Loss', linewidth=2, color='blue', linestyle='--', alpha=0.5)
ax3_twin.plot(history.history['val_loss'], label='Val Loss', linewidth=2, color='red', linestyle='--', alpha=0.5)
ax3.set_xlabel('Epochs', fontsize=12)
ax3.set_ylabel('Accuracy', fontsize=12, color='black')
ax3_twin.set_ylabel('Loss', fontsize=12, color='black')
ax3.set_title('Combined Training Progress', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, len(history.history['accuracy']))
ax3.set_ylim(0, 1.0)
ax3_twin.set_ylim(0, max(max(history.history['loss']), max(history.history['val_loss'])) * 1.1)

# Combine legends
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=9)

# Plot 4: Performance Metrics Bar Chart
ax4 = axes[1, 1]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
values = [accuracy, precision, recall, f1]
colors_metrics = ['#2ecc71', '#3498db', '#f39c12', '#9b59b6']
bars = ax4.bar(metrics, values, color=colors_metrics, alpha=0.8, edgecolor='black', linewidth=1.5)
ax4.set_ylabel('Percentage (%)', fontsize=12)
ax4.set_title('Model Performance Metrics', fontsize=14, fontweight='bold')
ax4.set_ylim(0, 100)
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, value in zip(bars, values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{value:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add target line (85.25% as reported)
ax4.axhline(y=85.25, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Target Accuracy (85.25%)')
ax4.legend(loc='lower right')

plt.suptitle('Monkeypox Severity Classification - Training Performance', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================================
# PART 9: CONFUSION MATRIX
# ============================================================================

print("\n[9] Confusion Matrix...")

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar_kws={'label': 'Number of Predictions'})
ax.set_title('Confusion Matrix for Monkeypox Severity Classification', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# ============================================================================
# PART 10: PER-CLASS PERFORMANCE
# ============================================================================

print("\n[10] Per-Class Performance...")

per_class_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [report[c]['precision'] * 100 for c in class_names],
    'Recall': [report[c]['recall'] * 100 for c in class_names],
    'F1-Score': [report[c]['f1-score'] * 100 for c in class_names],
    'Support': [report[c]['support'] for c in class_names]
})

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(class_names))
width = 0.25

bars1 = ax.bar(x - width, per_class_df['Precision'], width, label='Precision', color='#3498db', alpha=0.8)
bars2 = ax.bar(x, per_class_df['Recall'], width, label='Recall', color='#2ecc71', alpha=0.8)
bars3 = ax.bar(x + width, per_class_df['F1-Score'], width, label='F1-Score', color='#f39c12', alpha=0.8)

ax.set_xlabel('Severity Class', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 100)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nPer-Class Performance Summary:")
print("-" * 70)
print(per_class_df.to_string(index=False))
print("-" * 70)

# ============================================================================
# PART 11: ROC CURVES
# ============================================================================

print("\n[11] ROC Curves...")

plt.figure(figsize=(10, 8))
n_classes = len(class_names)
y_test_onehot = to_categorical(y_test, num_classes=n_classes)

for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_onehot[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{class_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves by Severity Class', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# PART 12: PRECISION-RECALL CURVES
# ============================================================================

print("\n[12] Precision-Recall Curves...")

plt.figure(figsize=(10, 8))

for i, class_name in enumerate(class_names):
    precision_curve, recall_curve, _ = precision_recall_curve(y_test_onehot[:, i], y_pred_proba[:, i])
    plt.plot(recall_curve, precision_curve, lw=2, label=class_name)

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves by Severity Class', fontsize=14, fontweight='bold')
plt.legend(loc='lower left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.show()

# ============================================================================
# PART 13: SAMPLE PREDICTIONS
# ============================================================================

print("\n[13] Sample Predictions...")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

indices = np.random.choice(len(X_test_seg), 8, replace=False)

for idx, ax in enumerate(axes):
    if idx < len(indices):
        i = indices[idx]
        img = X_test_seg[i]
        true_label = class_names[y_test[i]]
        pred_label = class_names[y_pred[i]]
        confidence = np.max(y_pred_proba[i]) * 100
        correct = y_test[i] == y_pred[i]

        ax.imshow(img)
        color = 'green' if correct else 'red'
        title = f'True: {true_label}\nPred: {pred_label}\nConf: {confidence:.1f}%'
        ax.set_title(title, fontsize=10, color=color, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================================
# PART 14: TRAINING SUMMARY TABLE
# ============================================================================

print("\n[14] Training Summary...")

# Create comprehensive results table
results_df = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Average Precision', 'Average Recall', 'Average F1-Score',
               'Best Validation Accuracy', 'Best Validation Loss', 'Total Epochs', 'Early Stopping Epoch'],
    'Value': [
        f'{accuracy:.2f}%',
        f'{precision:.2f}%',
        f'{recall:.2f}%',
        f'{f1:.2f}%',
        f'{best_val_acc*100:.2f}% (Epoch {best_epoch+1})',
        f'{best_val_loss:.4f} (Epoch {best_loss_epoch+1})',
        f'{len(history.history["accuracy"])}',
        f'{len(history.history["accuracy"])}'
    ]
})

print("\nTraining Summary:")
print("=" * 60)
print(results_df.to_string(index=False))
print("=" * 60)

# ============================================================================
# PART 15: SAVE RESULTS
# ============================================================================

print("\n[15] Saving Results...")

os.makedirs('results', exist_ok=True)

# Save model
model.save('results/monkeypox_severity_model.h5')
print("  Model saved to 'results/monkeypox_severity_model.h5'")

# Save metrics
per_class_df.to_csv('results/per_class_metrics.csv', index=False)
print("  Per-class metrics saved to 'results/per_class_metrics.csv'")

# Save predictions
predictions_df = pd.DataFrame({
    'True_Label': [class_names[y] for y in y_test],
    'Predicted_Label': [class_names[y] for y in y_pred],
    'Confidence': np.max(y_pred_proba, axis=1) * 100,
    'Correct': y_test == y_pred
})
predictions_df.to_csv('results/predictions.csv', index=False)
print("  Predictions saved to 'results/predictions.csv'")

# Save training history
history_df = pd.DataFrame({
    'Epoch': range(1, len(history.history['accuracy']) + 1),
    'Train_Accuracy': history.history['accuracy'],
    'Val_Accuracy': history.history['val_accuracy'],
    'Train_Loss': history.history['loss'],
    'Val_Loss': history.history['val_loss']
})
history_df.to_csv('results/training_history.csv', index=False)
print("  Training history saved to 'results/training_history.csv'")

# ============================================================================
# PART 16: FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(f"\nDataset Information:")
print(f"  • Total Images: {len(images)}")
print(f"  • Classes: {len(class_names)}")
print(f"  • Train/Val/Test Split: 70/15/15")

print(f"\nModel Performance:")
print(f"  • Test Accuracy:  {accuracy:.2f}%")
print(f"  • Avg Precision:  {precision:.2f}%")
print(f"  • Avg Recall:     {recall:.2f}%")
print(f"  • Avg F1-Score:   {f1:.2f}%")

print(f"\nBest Performing Class: {class_names[np.argmax([report[c]['f1-score'] for c in class_names])]}")
print(f"Worst Performing Class: {class_names[np.argmin([report[c]['f1-score'] for c in class_names])]}")

print(f"\nTraining Details:")
print(f"  • Total Epochs: {len(history.history['accuracy'])}")
print(f"  • Best Val Accuracy: {best_val_acc*100:.2f}% (Epoch {best_epoch+1})")
print(f"  • Best Val Loss: {best_val_loss:.4f} (Epoch {best_loss_epoch+1})")
print(f"  • Early Stopping: {'Yes' if len(history.history['accuracy']) < 100 else 'No'}")

print("\n" + "=" * 80)
print("TRAINING PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 80)

In [ ]:
"""
================================================================================
MONKEYPOX SEVERITY CLASSIFICATION - SEPARATE GRAPH VISUALIZATIONS
================================================================================
Each graph is plotted individually for better clarity and presentation
Dataset: Google Drive - mpoxdataset/ (70/15/15 split)
================================================================================
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import LabelEncoder
from scipy import stats
from tqdm import tqdm
import warnings
import glob
from google.colab import drive
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("=" * 80)
print("MONKEYPOX SEVERITY CLASSIFICATION - TRAINING PIPELINE")
print("=" * 80)
print(f"TensorFlow Version: {tf.__version__}")
print("=" * 80)

# ============================================================================
# PART 1: MOUNT GOOGLE DRIVE AND LOAD DATASET
# ============================================================================

print("\n[1] Mounting Google Drive...")
drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/mpoxdataset/'

if not os.path.exists(DATASET_PATH):
    print(f"Error: Dataset not found at {DATASET_PATH}")
    print("Please ensure your dataset is in the correct location:")
    print("  /content/drive/MyDrive/mpoxdataset/")
    print("  ├── 1_Macules/")
    print("  ├── 2_Papules/")
    print("  ├── 3_Vesicles/")
    print("  ├── 4_Pustules/")
    print("  ├── 5_Scubs/")
    print("  └── 6_Normal/")
    sys.exit(1)

class_names = ['Macules', 'Papules', 'Vesicles', 'Pustules', 'Scubs', 'Normal']
class_mapping = {
    '1_Macules': 0,
    '2_Papules': 1,
    '3_Vesicles': 2,
    '4_Pustules': 3,
    '5_Scubs': 4,
    '6_Normal': 5
}

print("\n[2] Loading Dataset...")
print(f"Dataset Path: {DATASET_PATH}")

images = []
labels = []
image_paths = []

for folder_name, label in class_mapping.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)

    if not os.path.exists(folder_path):
        print(f"Warning: Folder not found: {folder_path}")
        continue

    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
        image_files.extend(glob.glob(os.path.join(folder_path, ext)))

    print(f"  {folder_name}: {len(image_files)} images")

    for img_path in tqdm(image_files, desc=f"Loading {folder_name}"):
        try:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))
                images.append(img)
                labels.append(label)
                image_paths.append(img_path)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

images = np.array(images, dtype=np.float32) / 255.0
labels = np.array(labels)

print(f"\nDataset Loaded Successfully!")
print(f"  Total Images: {len(images)}")
print(f"  Image Shape: {images[0].shape}")
print(f"  Classes: {len(np.unique(labels))}")
print("\nClass Distribution:")
for i, class_name in enumerate(class_names):
    count = np.sum(labels == i)
    print(f"  {class_name}: {count} images ({count/len(labels)*100:.1f}%)")

# ============================================================================
# PART 3: DATA SPLITTING (70/15/15)
# ============================================================================

print("\n[3] Splitting Dataset (70% Train, 15% Validation, 15% Test)...")

# First split: 70% train, 30% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.30, stratify=labels, random_state=42
)

# Second split: 50% of temp for validation, 50% for test (15% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"\nSplit Summary:")
print(f"  Training samples:   {len(X_train)} ({len(X_train)/len(images)*100:.1f}%)")
print(f"  Validation samples: {len(X_val)} ({len(X_val)/len(images)*100:.1f}%)")
print(f"  Test samples:       {len(X_test)} ({len(X_test)/len(images)*100:.1f}%)")

# ============================================================================
# PART 4: THRESHOLD SEGMENTATION
# ============================================================================

class ThresholdSegmentation:
    """Threshold segmentation for isolating skin lesions."""

    def __init__(self, method='adaptive'):
        self.method = method

    def segment_image(self, image):
        """Apply threshold segmentation to isolate lesions."""
        if len(image.shape) == 3:
            gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        else:
            gray = (image * 255).astype(np.uint8)

        blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        if self.method == 'otsu':
            _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif self.method == 'adaptive':
            mask = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                        cv2.THRESH_BINARY, 11, 2)
        else:
            _, mask = cv2.threshold(blurred, 128, 255, cv2.THRESH_BINARY)

        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        if len(image.shape) == 3:
            segmented = cv2.bitwise_and((image * 255).astype(np.uint8),
                                       (image * 255).astype(np.uint8), mask=mask)
            segmented = segmented.astype(np.float32) / 255.0
        else:
            segmented = cv2.bitwise_and((image * 255).astype(np.uint8),
                                       (image * 255).astype(np.uint8), mask=mask)
            segmented = segmented.astype(np.float32) / 255.0

        return segmented

print("\n[4] Applying Threshold Segmentation...")
segmenter = ThresholdSegmentation(method='adaptive')

print("  Segmenting training set...")
X_train_seg = np.array([segmenter.segment_image(img) for img in tqdm(X_train, desc="  Training")])

print("  Segmenting validation set...")
X_val_seg = np.array([segmenter.segment_image(img) for img in tqdm(X_val, desc="  Validation")])

print("  Segmenting test set...")
X_test_seg = np.array([segmenter.segment_image(img) for img in tqdm(X_test, desc="  Test")])

# ============================================================================
# PART 5: CNN MODEL ARCHITECTURE
# ============================================================================

class MonkeypoxSeverityCNN:
    """Deep CNN for monkeypox severity classification."""

    def __init__(self, input_shape=(224, 224, 3), num_classes=6):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = None

    def build_model(self):
        """Build the CNN model."""
        inputs = layers.Input(shape=self.input_shape)

        # First block
        x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.25)(x)

        # Second block
        x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.25)(x)

        # Third block
        x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.3)(x)

        # Fourth block
        x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.3)(x)

        # Fifth block
        x = layers.Conv2D(512, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(512, (3, 3), padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.5)(x)

        # Dense layers
        x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)

        # Output
        outputs = layers.Dense(self.num_classes, activation='softmax')(x)

        self.model = models.Model(inputs=inputs, outputs=outputs)
        return self.model

    def compile_model(self, learning_rate=0.001):
        """Compile the model with metrics."""
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return self.model

# ============================================================================
# PART 6: TRAINING
# ============================================================================

print("\n[5] Building and Training Model...")

# Build model
cnn = MonkeypoxSeverityCNN(input_shape=(224, 224, 3), num_classes=len(class_names))
model = cnn.build_model()
model = cnn.compile_model(learning_rate=0.001)

# Model summary
model.summary()

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Train model
print("\n[6] Training Model...")
history = model.fit(
    X_train_seg, y_train,
    validation_data=(X_val_seg, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# ============================================================================
# PART 7: EVALUATION
# ============================================================================

print("\n[7] Evaluating Model...")

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_seg, y_test, verbose=0)

# Predictions
y_pred_proba = model.predict(X_test_seg)
y_pred = np.argmax(y_pred_proba, axis=1)

# Classification report
report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)

# Calculate metrics
accuracy = test_accuracy * 100
precision = np.mean([report[c]['precision'] for c in class_names]) * 100
recall = np.mean([report[c]['recall'] for c in class_names]) * 100
f1 = np.mean([report[c]['f1-score'] for c in class_names]) * 100

print(f"\n{'='*60}")
print("FINAL PERFORMANCE METRICS")
print(f"{'='*60}")
print(f"Test Accuracy:    {accuracy:.2f}%")
print(f"Average Precision: {precision:.2f}%")
print(f"Average Recall:    {recall:.2f}%")
print(f"Average F1-Score:  {f1:.2f}%")
print(f"{'='*60}")

# ============================================================================
# PART 8: GRAPH 1 - TRAINING AND VALIDATION ACCURACY
# ============================================================================

print("\n[8] Graph 1: Training and Validation Accuracy...")

plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=3, color='blue', marker='o', markersize=4, markevery=5)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=3, color='red', marker='s', markersize=4, markevery=5)
plt.axhline(y=test_accuracy, color='green', linestyle='--', label=f'Test Accuracy: {test_accuracy*100:.2f}%', linewidth=2)
plt.xlabel('Epochs', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy', fontsize=14, fontweight='bold')
plt.title('Training and Validation Accuracy', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xlim(0, len(history.history['accuracy']))
plt.ylim(0, 1.0)

# Add best accuracy annotation
best_val_acc = max(history.history['val_accuracy'])
best_epoch = history.history['val_accuracy'].index(best_val_acc)
plt.annotate(f'Best: {best_val_acc*100:.2f}%\nEpoch {best_epoch+1}',
            xy=(best_epoch, best_val_acc),
            xytext=(best_epoch + 5, best_val_acc - 0.1),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

# ============================================================================
# PART 9: GRAPH 2 - TRAINING AND VALIDATION LOSS
# ============================================================================

print("\n[9] Graph 2: Training and Validation Loss...")

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss', linewidth=3, color='blue', marker='o', markersize=4, markevery=5)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=3, color='red', marker='s', markersize=4, markevery=5)
plt.axhline(y=test_loss, color='green', linestyle='--', label=f'Test Loss: {test_loss:.4f}', linewidth=2)
plt.xlabel('Epochs', fontsize=14, fontweight='bold')
plt.ylabel('Loss', fontsize=14, fontweight='bold')
plt.title('Training and Validation Loss', fontsize=16, fontweight='bold')
plt.legend(loc='upper right', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xlim(0, len(history.history['loss']))
plt.ylim(0, max(max(history.history['loss']), max(history.history['val_loss'])) * 1.1)

# Add best loss annotation
best_val_loss = min(history.history['val_loss'])
best_loss_epoch = history.history['val_loss'].index(best_val_loss)
plt.annotate(f'Best: {best_val_loss:.4f}\nEpoch {best_loss_epoch+1}',
            xy=(best_loss_epoch, best_val_loss),
            xytext=(best_loss_epoch - 10, best_val_loss + 0.3),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

# ============================================================================
# PART 10: GRAPH 3 - COMBINED ACCURACY AND LOSS
# ============================================================================

print("\n[10] Graph 3: Combined Accuracy and Loss...")

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot accuracy on left y-axis
color = 'tab:blue'
ax1.set_xlabel('Epochs', fontsize=14, fontweight='bold')
ax1.set_ylabel('Accuracy', color=color, fontsize=14, fontweight='bold')
ax1.plot(history.history['accuracy'], label='Train Acc', linewidth=3, color=color, marker='o', markersize=4, markevery=5)
ax1.plot(history.history['val_accuracy'], label='Val Acc', linewidth=3, color='tab:cyan', marker='s', markersize=4, markevery=5)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1.0)

# Create second y-axis for loss
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Loss', color=color, fontsize=14, fontweight='bold')
ax2.plot(history.history['loss'], label='Train Loss', linewidth=3, color=color, linestyle='--', marker='o', markersize=4, markevery=5)
ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=3, color='tab:orange', linestyle='--', marker='s', markersize=4, markevery=5)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0, max(max(history.history['loss']), max(history.history['val_loss'])) * 1.1)

# Title
plt.title('Combined Training Progress', fontsize=16, fontweight='bold')

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=11)

plt.tight_layout()
plt.show()

# ============================================================================
# PART 11: GRAPH 4 - PERFORMANCE METRICS BAR CHART
# ============================================================================

print("\n[11] Graph 4: Performance Metrics...")

plt.figure(figsize=(10, 7))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
values = [accuracy, precision, recall, f1]
colors_metrics = ['#2ecc71', '#3498db', '#f39c12', '#9b59b6']
bars = plt.bar(metrics, values, color=colors_metrics, alpha=0.8,
               edgecolor='black', linewidth=2, width=0.6)
plt.ylabel('Percentage (%)', fontsize=14, fontweight='bold')
plt.title('Model Performance Metrics', fontsize=16, fontweight='bold')
plt.ylim(0, 100)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, value in zip(bars, values):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 1.5,
            f'{value:.2f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

# Add target line (85.25% as reported)
plt.axhline(y=85.25, color='red', linestyle='--', linewidth=2.5, alpha=0.7,
           label=f'Target Accuracy (85.25%)')
plt.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

# ============================================================================
# PART 12: GRAPH 5 - CONFUSION MATRIX
# ============================================================================

print("\n[12] Graph 5: Confusion Matrix...")

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Number of Predictions', 'shrink': 0.8},
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Confusion Matrix for Monkeypox Severity Classification', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=14, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

# ============================================================================
# PART 13: GRAPH 6 - PER-CLASS PERFORMANCE
# ============================================================================

print("\n[13] Graph 6: Per-Class Performance...")

per_class_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [report[c]['precision'] * 100 for c in class_names],
    'Recall': [report[c]['recall'] * 100 for c in class_names],
    'F1-Score': [report[c]['f1-score'] * 100 for c in class_names],
    'Support': [report[c]['support'] for c in class_names]
})

plt.figure(figsize=(12, 7))
x = np.arange(len(class_names))
width = 0.25

bars1 = plt.bar(x - width, per_class_df['Precision'], width, label='Precision',
                color='#3498db', alpha=0.8, edgecolor='black', linewidth=1.5)
bars2 = plt.bar(x, per_class_df['Recall'], width, label='Recall',
                color='#2ecc71', alpha=0.8, edgecolor='black', linewidth=1.5)
bars3 = plt.bar(x + width, per_class_df['F1-Score'], width, label='F1-Score',
                color='#f39c12', alpha=0.8, edgecolor='black', linewidth=1.5)

plt.xlabel('Severity Class', fontsize=14, fontweight='bold')
plt.ylabel('Percentage (%)', fontsize=14, fontweight='bold')
plt.title('Per-Class Performance Metrics', fontsize=16, fontweight='bold')
plt.xticks(x, class_names, rotation=45, ha='right')
plt.legend(loc='upper right', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.ylim(0, 100)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# ============================================================================
# PART 14: GRAPH 7 - ROC CURVES
# ============================================================================

print("\n[14] Graph 7: ROC Curves...")

plt.figure(figsize=(10, 8))
n_classes = len(class_names)
y_test_onehot = to_categorical(y_test, num_classes=n_classes)

colors_roc = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
for i, class_name in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_test_onehot[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=3, color=colors_roc[i], label=f'{class_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2.5, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=14, fontweight='bold')
plt.title('ROC Curves by Severity Class', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# PART 15: GRAPH 8 - PRECISION-RECALL CURVES
# ============================================================================

print("\n[15] Graph 8: Precision-Recall Curves...")

plt.figure(figsize=(10, 8))
colors_pr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
for i, class_name in enumerate(class_names):
    precision_curve, recall_curve, _ = precision_recall_curve(y_test_onehot[:, i], y_pred_proba[:, i])
    plt.plot(recall_curve, precision_curve, lw=3, color=colors_pr[i], label=class_name)

plt.xlabel('Recall', fontsize=14, fontweight='bold')
plt.ylabel('Precision', fontsize=14, fontweight='bold')
plt.title('Precision-Recall Curves by Severity Class', fontsize=16, fontweight='bold')
plt.legend(loc='lower left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.show()

# ============================================================================
# PART 16: GRAPH 9 - SAMPLE PREDICTIONS
# ============================================================================

print("\n[16] Graph 9: Sample Predictions...")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

indices = np.random.choice(len(X_test_seg), 8, replace=False)

for idx, ax in enumerate(axes):
    if idx < len(indices):
        i = indices[idx]
        img = X_test_seg[i]
        true_label = class_names[y_test[i]]
        pred_label = class_names[y_pred[i]]
        confidence = np.max(y_pred_proba[i]) * 100
        correct = y_test[i] == y_pred[i]

        ax.imshow(img)
        color = 'green' if correct else 'red'
        title = f'True: {true_label}\nPred: {pred_label}\nConf: {confidence:.1f}%'
        ax.set_title(title, fontsize=11, color=color, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================================
# PART 17: GRAPH 10 - CLASS DISTRIBUTION
# ============================================================================

print("\n[17] Graph 10: Class Distribution...")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

datasets = [(y_train, 'Training Set', '#3498db'),
            (y_val, 'Validation Set', '#2ecc71'),
            (y_test, 'Test Set', '#f39c12')]

for idx, (data, title, color) in enumerate(datasets):
    unique, counts = np.unique(data, return_counts=True)
    bars = axes[idx].bar(class_names, counts, color=color, alpha=0.8, edgecolor='black', linewidth=1.5)
    axes[idx].set_title(f'{title} Distribution', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Class', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Count', fontsize=12, fontweight='bold')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(True, alpha=0.3, axis='y')

    # Add value labels
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                      str(count), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Class Distribution Across Dataset Splits', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================================
# PART 18: SAVE RESULTS
# ============================================================================

print("\n[18] Saving Results...")

os.makedirs('results', exist_ok=True)

# Save model
model.save('results/monkeypox_severity_model.h5')
print("  Model saved to 'results/monkeypox_severity_model.h5'")

# Save metrics
per_class_df.to_csv('results/per_class_metrics.csv', index=False)
print("  Per-class metrics saved to 'results/per_class_metrics.csv'")

# Save predictions
predictions_df = pd.DataFrame({
    'True_Label': [class_names[y] for y in y_test],
    'Predicted_Label': [class_names[y] for y in y_pred],
    'Confidence': np.max(y_pred_proba, axis=1) * 100,
    'Correct': y_test == y_pred
})
predictions_df.to_csv('results/predictions.csv', index=False)
print("  Predictions saved to 'results/predictions.csv'")

# Save training history
history_df = pd.DataFrame({
    'Epoch': range(1, len(history.history['accuracy']) + 1),
    'Train_Accuracy': history.history['accuracy'],
    'Val_Accuracy': history.history['val_accuracy'],
    'Train_Loss': history.history['loss'],
    'Val_Loss': history.history['val_loss']
})
history_df.to_csv('results/training_history.csv', index=False)
print("  Training history saved to 'results/training_history.csv'")

# ============================================================================
# PART 19: FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(f"\nDataset Information:")
print(f"  • Total Images: {len(images)}")
print(f"  • Classes: {len(class_names)}")
print(f"  • Train/Val/Test Split: 70/15/15")

print(f"\nModel Performance:")
print(f"  • Test Accuracy:  {accuracy:.2f}%")
print(f"  • Avg Precision:  {precision:.2f}%")
print(f"  • Avg Recall:     {recall:.2f}%")
print(f"  • Avg F1-Score:   {f1:.2f}%")

print(f"\nBest Performing Class: {class_names[np.argmax([report[c]['f1-score'] for c in class_names])]}")
print(f"Worst Performing Class: {class_names[np.argmin([report[c]['f1-score'] for c in class_names])]}")

print(f"\nTraining Details:")
print(f"  • Total Epochs: {len(history.history['accuracy'])}")
print(f"  • Best Val Accuracy: {best_val_acc*100:.2f}% (Epoch {best_epoch+1})")
print(f"  • Best Val Loss: {best_val_loss:.4f} (Epoch {best_loss_epoch+1})")
print(f"  • Early Stopping: {'Yes' if len(history.history['accuracy']) < 100 else 'No'}")

print("\nGraphs Generated (10 Separate Visualizations):")
print("  1. Training and Validation Accuracy")
print("  2. Training and Validation Loss")
print("  3. Combined Accuracy and Loss (Dual Axis)")
print("  4. Performance Metrics Bar Chart")
print("  5. Confusion Matrix")
print("  6. Per-Class Performance")
print("  7. ROC Curves")
print("  8. Precision-Recall Curves")
print("  9. Sample Predictions")
print("  10. Class Distribution")

print("\n" + "=" * 80)
print("TRAINING PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 80)